# Normal (Gaussian) Distribution

The **Normal** (Gaussian) distribution is the most important distribution in statistics.
Its famous **bell curve** appears everywhere due to the Central Limit Theorem.

| Property | Value |
|----------|-------|
| **Notation** | $X \sim N(\mu, \sigma^2)$ |
| **Parameters** | $\mu$ = mean, $\sigma^2$ = variance |
| **Support** | $(-\infty, \infty)$ |
| **PDF** | $f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$ |
| **CDF** | No closed form — use tables or `scipy` |
| **Expectation** | $E[X] = \mu$ |
| **Variance** | $\text{Var}(X) = \sigma^2$ |

**Standard Normal:** $Z \sim N(0, 1)$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

print("Libraries loaded!")

---
## 1. The Bell Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Different means
ax = axes[0]
x = np.linspace(-8, 12, 300)
for mu, color in [(-2, '#e74c3c'), (0, '#3498db'), (3, '#2ecc71'), (5, '#9b59b6')]:
    rv = stats.norm(mu, 1)
    ax.plot(x, rv.pdf(x), color=color, linewidth=2.5, label=f'μ={mu}, σ=1')
ax.set_title('Varying Mean (μ)', fontsize=13, fontweight='bold')
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('f(x)', fontsize=12)
ax.legend(fontsize=10)

# Different variances
ax = axes[1]
x = np.linspace(-10, 10, 300)
for sigma, color in [(0.5, '#e74c3c'), (1, '#3498db'), (2, '#2ecc71'), (4, '#9b59b6')]:
    rv = stats.norm(0, sigma)
    ax.plot(x, rv.pdf(x), color=color, linewidth=2.5, label=f'μ=0, σ={sigma}')
ax.set_title('Varying Standard Deviation (σ)', fontsize=13, fontweight='bold')
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('f(x)', fontsize=12)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

---
## 2. The 68-95-99.7 Rule

For $X \sim N(\mu, \sigma^2)$:
- $P(\mu - \sigma < X < \mu + \sigma) \approx 68.27\%$
- $P(\mu - 2\sigma < X < \mu + 2\sigma) \approx 95.45\%$
- $P(\mu - 3\sigma < X < \mu + 3\sigma) \approx 99.73\%$

In [ ]:
rv = stats.norm(0, 1)
x = np.linspace(-4, 4, 500)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(x, rv.pdf(x), 'k-', linewidth=2.5)

# Shade regions
colors = ['#3498db', '#2ecc71', '#e74c3c']
alphas = [0.5, 0.35, 0.2]
labels = ['±1σ: 68.27%', '±2σ: 95.45%', '±3σ: 99.73%']

for k, color, alpha, label in zip([3, 2, 1], colors[::-1], alphas[::-1], labels[::-1]):
    mask = (x >= -k) & (x <= k)
    ax.fill_between(x[mask], rv.pdf(x[mask]), alpha=alpha, color=color, label=label)

ax.set_title('The 68-95-99.7 Rule (Standard Normal)', fontsize=14, fontweight='bold')
ax.set_xlabel('x (in units of σ)', fontsize=12)
ax.set_ylabel('f(x)', fontsize=12)
ax.set_xticks([-3, -2, -1, 0, 1, 2, 3])
ax.set_xticklabels(['-3σ', '-2σ', '-σ', 'μ', '+σ', '+2σ', '+3σ'], fontsize=11)
ax.legend(fontsize=12, loc='upper right')
plt.tight_layout()
plt.show()

# Verify
for k in [1, 2, 3]:
    p = rv.cdf(k) - rv.cdf(-k)
    print(f"P({-k}σ < X < {k}σ) = {p:.6f} = {p*100:.2f}%")

---
## 3. Z-Scores — Standardization

Any normal can be converted to the standard normal:

$$Z = \frac{X - \mu}{\sigma} \sim N(0, 1)$$

This means: $P(X \leq x) = P\left(Z \leq \frac{x - \mu}{\sigma}\right) = \Phi\left(\frac{x - \mu}{\sigma}\right)$

In [ ]:
# Example: exam scores N(70, 10²)
mu, sigma = 70, 10

# What fraction scored above 85?
z = (85 - mu) / sigma
p_above_85 = 1 - stats.norm.cdf(z)

print(f"Exam scores ~ N(μ={mu}, σ={sigma})")
print(f"  Score of 85: z-score = (85-70)/10 = {z}")
print(f"  P(X > 85) = P(Z > {z}) = {p_above_85:.4f} = {p_above_85*100:.2f}%")
print()

# What score is the 90th percentile?
percentile_90 = stats.norm.ppf(0.90, loc=mu, scale=sigma)
print(f"  90th percentile: {percentile_90:.1f}")
print(f"  z-score for 90th: {stats.norm.ppf(0.90):.4f}")

# Visual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original scale
ax = axes[0]
x = np.linspace(30, 110, 300)
rv = stats.norm(mu, sigma)
ax.plot(x, rv.pdf(x), 'b-', linewidth=2.5)
ax.fill_between(x[x >= 85], rv.pdf(x[x >= 85]), alpha=0.5, color='red',
               label=f'P(X > 85) = {p_above_85:.4f}')
ax.axvline(85, color='red', linestyle='--', linewidth=1.5)
ax.set_title(f'Original: N({mu}, {sigma}²)', fontsize=12, fontweight='bold')
ax.set_xlabel('Score', fontsize=11)
ax.legend(fontsize=10)

# Z-score scale
ax = axes[1]
z_vals = np.linspace(-4, 4, 300)
rv_std = stats.norm(0, 1)
ax.plot(z_vals, rv_std.pdf(z_vals), 'b-', linewidth=2.5)
ax.fill_between(z_vals[z_vals >= z], rv_std.pdf(z_vals[z_vals >= z]), alpha=0.5, color='red',
               label=f'P(Z > {z}) = {p_above_85:.4f}')
ax.axvline(z, color='red', linestyle='--', linewidth=1.5)
ax.set_title(f'Standardized: N(0, 1)', fontsize=12, fontweight='bold')
ax.set_xlabel('z-score', fontsize=11)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

---
## 4. Properties of the Normal Distribution

| Property | Formula |
|----------|---------|
| **Linear transform** | If $X \sim N(\mu, \sigma^2)$, then $aX + b \sim N(a\mu + b, a^2\sigma^2)$ |
| **Sum** | $X + Y \sim N(\mu_X + \mu_Y, \sigma_X^2 + \sigma_Y^2)$ if independent |
| **Symmetry** | $P(X < \mu - a) = P(X > \mu + a)$ |

In [ ]:
# Sum of normals is normal
np.random.seed(42)
n = 100000

X = np.random.normal(3, 2, size=n)  # N(3, 4)
Y = np.random.normal(5, 3, size=n)  # N(5, 9)
Z = X + Y  # should be N(8, 13)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(Z, bins=100, density=True, color='#3498db', edgecolor='white', alpha=0.7,
        label=f'X + Y simulated (μ≈{Z.mean():.2f}, σ²≈{Z.var():.2f})')

x_plot = np.linspace(-5, 25, 300)
rv_theory = stats.norm(8, np.sqrt(13))
ax.plot(x_plot, rv_theory.pdf(x_plot), 'r-', linewidth=2.5,
        label=f'N(8, 13) theory')

ax.set_title('Sum of Independent Normals is Normal', fontsize=13, fontweight='bold')
ax.set_xlabel('x', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"Theory: N(3+5, 4+9) = N(8, 13)")
print(f"Simulation: μ = {Z.mean():.4f}, σ² = {Z.var():.4f}")

---
## 5. Python: scipy.stats.norm

In [ ]:
from scipy.stats import norm

mu, sigma = 100, 15  # IQ scores
rv = norm(loc=mu, scale=sigma)

print(f"IQ scores ~ N({mu}, {sigma}²):")
print(f"  P(IQ > 130)      = {rv.sf(130):.4f}  ({rv.sf(130)*100:.2f}%)")
print(f"  P(85 < IQ < 115) = {rv.cdf(115) - rv.cdf(85):.4f}  ({(rv.cdf(115) - rv.cdf(85))*100:.2f}%)")
print(f"  P(IQ < 70)       = {rv.cdf(70):.4f}  ({rv.cdf(70)*100:.2f}%)")
print(f"\n  Median = {rv.median():.0f}")
print(f"  95th percentile = {rv.ppf(0.95):.1f}")
print(f"  99th percentile = {rv.ppf(0.99):.1f}")
print(f"\n  Samples: {rv.rvs(size=8, random_state=42).astype(int)}")

---
## Summary

| Property | Formula |
|----------|---------|
| PDF | $f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-(x-\mu)^2/2\sigma^2}$ |
| $E[X]$ | $\mu$ |
| $\text{Var}(X)$ | $\sigma^2$ |
| Z-score | $Z = (X - \mu)/\sigma \sim N(0,1)$ |
| 68-95-99.7 | Within 1σ/2σ/3σ of the mean |
| Sum | $N(\mu_1, \sigma_1^2) + N(\mu_2, \sigma_2^2) = N(\mu_1+\mu_2, \sigma_1^2+\sigma_2^2)$ |
| Python | `scipy.stats.norm(loc=μ, scale=σ)` |

**Next:** Approximating Binomial — connecting discrete and continuous.